# Lab | LangChain Med

## Objectives

- continue on with lesson 2' example, use different datasets to test what we did in class. Some datasets are suggested in the notebook but feel free to scout other datasets on HuggingFace or Kaggle.
- Find another model on Hugging Face and compare it.
- Modify the prompt to fit your selected dataset.

In [1]:
import numpy as np
import pandas as pd

## Load the Dataset
As you can see the notebook is ready to work with three different Datasets. Just uncomment the lines of the Dataset you want to use.

I selected Datasets with News. Two of them have just a brief decription of the news, but the other contains the full text.

As we are working in a free and limited space, I limited the number of news to use with the variable MAX_NEWS. Feel free to pull more if you have memory available.

The name of the field containing the text of the new is stored in the variable *DOCUMENT* and the metadata in *TOPIC*

In [2]:
# news = pd.read_csv('/kaggle/input/topic-labeled-news-dataset/labelled_newscatcher_dataset.csv', sep=';')
# MAX_NEWS = 1000
# DOCUMENT="title"
# TOPIC="topic"

#news = pd.read_csv('/kaggle/input/bbc-news/bbc_news.csv')
#MAX_NEWS = 1000
#DOCUMENT="description"
#TOPIC="title"

news = pd.read_csv('/content/articles.csv')
MAX_NEWS = 100
DOCUMENT="Article Body"
TOPIC="Article Header"

# news = "PICK A DATASET" #Ideally pick one from the commented ones above

For testing a RAG/vector-DB workflow, full text articles are much better because:
  - embeddings have richerr context - retrieval quality is easier to judge
  - the llm can answer more than "headline-level" quaestions
  - chunking actually matters (good to practice)

  Therefore MIT AI News Published till 2023

ChromaDB requires that the data has a unique identifier. We can make it with this statement, which will create a new column called **Id**.


In [3]:
news["id"] = news.index
news.head()

,Unnamed: 0,Published Date,Author,Source,Article Header,Sub_Headings,Article Body,Url,id
0,0,"July 7, 2023",Adam Zewe,MIT News Office,Learning the language of molecules to predict ...,This AI system only needs a small amount of da...,['Discovering new materials and drugs typicall...,https://news.mit.edu/2023/learning-language-mo...,0
1,1,"July 6, 2023",Alex Ouyang,Abdul Latif Jameel Clinic for Machine Learning...,MIT scientists build a system that can generat...,"BioAutoMATED, an open-source, automated machin...",['Is it possible to build machine-learning mod...,https://news.mit.edu/2023/bioautomated-open-so...,1
2,2,"June 30, 2023",Jennifer Michalowski,McGovern Institute for Brain Research,"When computer vision works more like a brain, ...",Training artificial neural networks with data ...,"['From cameras to self-driving cars, many of t...",https://news.mit.edu/2023/when-computer-vision...,2
3,3,"June 30, 2023",Mary Beth Gallagher,School of Engineering,Educating national security leaders on artific...,"Experts from MIT’s School of Engineering, Schw...",['Understanding artificial intelligence and ho...,https://news.mit.edu/2023/educating-national-s...,3
4,4,"June 30, 2023",Adam Zewe,MIT News Office,Researchers teach an AI to write better chart ...,A new dataset can help scientists develop auto...,['Chart captions that explain complex trends a...,https://news.mit.edu/2023/researchers-chart-ca...,4


In [4]:
#Because it is just a course we select a small portion of News.
subset_news = news.sample(n=min(MAX_NEWS, len(news)), random_state=42).reset_index(drop=True)
print("Subset size:", len(subset_news))
subset_news.head()
# avoids bias from dataset ordering (eg. first rows all same year/topic); still reproducible (random_state=42)


Subset size: 100


,Unnamed: 0,Published Date,Author,Source,Article Header,Sub_Headings,Article Body,Url,id
0,528,"June 17, 2019",Rachel Gordon,MIT CSAIL,Teaching artificial intelligence to connect se...,MIT CSAIL system can learn to see by touching ...,"['In Canadian author Margaret Atwood’s book ""B...",https://news.mit.edu/2019/teaching-ai-to-conne...,528
1,914,"February 1, 2005","Andrew Spann, Class of 2007",NaN,A robot high for Archimedes Pi,NaN,['The Mobile Autonomous Systems Laboratory (Ma...,https://news.mit.edu/2005/iap-maslab,914
2,587,"February 1, 2019",Annie Young,Institute for Medical Engineering and Science,MIMIC Chest X-Ray database to provide research...,A new database of images could pave a path for...,"['Computer vision, or the method of giving mac...",https://news.mit.edu/2019/mimic-chest-x-ray-da...,587
3,31,"May 17, 2023",Peter Dizikes,MIT News Office,An AI challenge only humans can solve,"In their new book, “Power and Progress,” Daron...",['The Dark Ages were not entirely dark. Advanc...,https://news.mit.edu/2023/power-and-progress-b...,31
4,136,"June 30, 2022",Adam Zewe,MIT News Office,Building explainability into the components of...,Researchers develop tools to help data scienti...,['Explanation methods that help users understa...,https://news.mit.edu/2022/explainability-machi...,136


## Import and configure the Vector Database
I'm going to use ChromaDB, the most popular OpenSource embedding Database.

First we need to import ChromaDB, and after that import the **Settings** class from **chromadb.config** module. This class allows us to change the setting for the ChromaDB system, and customize its behavior.

In [5]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 7.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 10.8 MB/s 

In [6]:
import chromadb
from chromadb.config import Settings

Now we need to create the seetings object calling the **Settings** function imported previously. We store the object in the variable **settings_chroma**.

Is necessary to inform two parameters
* chroma_db_impl. Here we specify the database implementation and the format how store the data. I choose ***duckdb***, because his high-performace. It operate primarly in memory. And is fully compatible with SQL. The store format ***parquet*** is good for tabular data. With good compression rates and performance.

* persist_directory: It just contains the directory where the data will be stored. Is possible work without a directory and the data will be stored in memory without persistece, but Kaggle dosn't support that.

In [7]:
chroma_client = chromadb.PersistentClient(path="/path/to/persist/directory")

## Filling and Querying the ChromaDB Database
The Data in ChromaDB is stored in collections. If the collection exist we need to delete it.

In the next lines, we are creating the collection by calling the ***create_collection*** function in the ***chroma_client*** created above.

In [8]:
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# 1) normalize docs
def normalize_doc(x):
    if isinstance(x, list):
        return " ".join(map(str, x))
    return str(x)

subset_news[DOCUMENT] = subset_news[DOCUMENT].apply(normalize_doc)


# 2) fresh collection with embeddings ON
collection_name = "news_collection_rali" # choosing a name for Croma DB collection for future use
# collection is like a table in SQL or an index in Elasticsearch; where embeddied documents are stored

# get current collection names
existing = [c.name for c in chroma_client.list_collections()]
# chroma_client.list_collections() returns all collections currently stored in your Chroma database
# each item c is a collection object, and c.name is its name
# the list comprehension builds a simple Python list of those names

if collection_name in existing:
    chroma_client.delete_collection(name=collection_name)
# this checks whether "news_collection_rali" already exists
# if it does you delete it
# we need a fresh, empty collection

embedding_fn = SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

collection = chroma_client.create_collection(
    name=collection_name,
    embedding_function=embedding_fn
)
print("Collection ready:", collection.name)
# now we create a prand-new empty collection with that name
# the returned object is stored in collection, which we will use later to add() documents and quary them



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Collection ready: news_collection_rali


It's time to add the data to the collection. Using the function ***add*** we need to inform, at least ***documents***, ***metadatas*** and ***ids***.
* In the **document** we store the big text, it's a different column in each Dataset.
* In **metadatas**, we can informa a list of topics.
* In **id** we need to inform an unique identificator for each row. It MUST be unique! I'm creating the ID using the range of MAX_NEWS.


In [9]:
# ADDING DATA to ChromaDB collection
# 1.Prepare documents (texts)
# 2.Prepare metadata (topics/titles/labels)
# 3. Prepare unique IDs (must be strings)
n = len(subset_news)

collection.add(
    documents=subset_news[DOCUMENT].astype(str).tolist(),
    metadatas=[{"topic": str(t)} for t in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(n)],
)

print("Added", n, "documents.")
print("Stored docs:", collection.count())



Added 100 documents.
Stored docs: 100


In [10]:
results = collection.query(
    query_texts=["energy efficient AI hardware"],
    n_results=5,
    include=["documents", "metadatas", "distances", "embeddings"]
)

print(type(results["embeddings"][0][0]))  # should be a list/np array


# query_texts=["laptop"] - we give Chrome a list of queries. Even if it's one query, it must be a list
# N_results=10 Returns the top 10 nearest neighbours (most similar docs)

print(results)

<class 'numpy.ndarray'>
{'ids': [['id83', 'id63', 'id41', 'id37', 'id24']], 'embeddings': [array([[-0.10799517,  0.02798677, -0.00788515, ..., -0.03075094,
        -0.04895537,  0.02842336],
       [-0.05633797, -0.05315839, -0.0020007 , ..., -0.03311954,
         0.04944369,  0.03729889],
       [-0.02973429,  0.00335998,  0.03080417, ...,  0.02675907,
        -0.00282881, -0.00092606],
       [-0.08310951, -0.06046108,  0.00073832, ...,  0.0025191 ,
        -0.0749605 , -0.02807314],
       [-0.07258982,  0.01169531, -0.03557327, ..., -0.13018778,
        -0.03307239, -0.00806708]])], 'documents': [["['The MIT AI Hardware Program is a new academia and industry collaboration aimed at defining and developing translational technologies in hardware and software for the AI and quantum age. A collaboration between the MIT School of Engineering and MIT Schwarzman College of Computing, involving the Microsystems Technologies Laboratories and programs and units in the college, the cross-disci

## Vector MAP

In [11]:
import matplotlib.pyplot as plt # for plotting later
from sklearn.decomposition import PCA # is for reducing high-dim embeddings to 2D/3D for visualization

In [12]:
# get a specific document + embedding from Chroma

getado = collection.get(
    ids=["id141"],
    include=["documents", "embeddings", "metadatas"]
)

print("Keys:", getado.keys())
print("IDs returned:", getado["ids"])
print("Num docs:", len(getado["documents"]))
print("Num embeddings:", len(getado["embeddings"]) if getado["embeddings"] is not None else None)
print("Metadatas:", getado["metadatas"])
# we are asking Chroma: "give me the stored text and the embeddin for id141"

Keys: dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])
IDs returned: []
Num docs: 0
Num embeddings: 0
Metadatas: []


In [13]:
# extract embeddings and documents

word_vectors = getado["embeddings"] # becomes the embeddings returded from Chroma
word_list = getado["documents"] # becomes the text returned by Chroma
word_vectors

array([], dtype=float64)

Once we have our information inside the Database we can query It, and ask for data that matches our needs. The search is done inside the content of the document, and it dosn't look for the exact word, or phrase. The results will be based on the similarity between the search terms and the content of documents.

The metadata is not used in the search, but they can be utilized for filtering or refining the results after the initial search.


## Loading the model and creating the prompt
TRANSFORMERS!!
Time to use the library **transformers**, the most famous library from [hugging face](https://huggingface.co/) for working with language models.

We are importing:
* **Autotokenizer**: It is a utility class for tokenizing text inputs that are compatible with various pre-trained language models.# AutoTokenizer: a helper class that automatically loads the correct tokenizer for a given pretrained model, and converts text into tokens the model understands.

* **AutoModelForCasualLLM**: it provides an interface to pre-trained language models specifically designed for language generation tasks using causal language modeling (e.g., GPT models), or the model used in this notebook ***databricks/dolly-v2-3b***.# an interface for pretrained causal language models (decoder-only models like GPT). These models generate text by predicting the next token in a sequence.

* **pipeline**: provides a simple interface for performing various natural language processing (NLP) tasks, such as text generation (our case) or text classification. # a high-level wrapper that makes it easy to run common NLP tasks (in our case, text generation) with only a few lines of code.

The model selected is [dolly-v2-3b](https://huggingface.co/databricks/dolly-v2-3b), the smallest Dolly model. It has 3billion paramaters, more than enough for our sample, and works much better than GPT2.

Please, feel free to test [different Models](https://huggingface.co/models?pipeline_tag=text-generation&sort=trending), you need to search for NLP models trained for text-generation. My recomendation is choose "small" models, or we will run out of memory in kaggle.  
a high-level wrapper that makes it easy to run common NLP tasks (in our case, text generation) with only a few lines of code.

In [14]:
import gc, torch

# delete big objects if they exist
for name in ["lm_model", "model_a", "model_b", "pipe", "pipe_a", "pipe_b", "tokenizer_a", "tokenizer_b"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print("Freed GPU cache.")


Freed GPU cache.


In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "databricks/dolly-v2-3b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)



tokenizer_config.json:   0%|          | 0.00/450 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/819 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.68G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.68G [00:00<?, ?B/s]

The next step is to initialize the pipeline using the objects created above.

The model's response is limited to 256 tokens, for this project I'm not interested in a longer response, but it can easily be extended to whatever length you want.

Setting ***device_map*** to ***auto*** we are instructing the model to automaticaly select the most appropiate device: CPU or GPU for processing the text generation.  

In [16]:
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    device_map="auto",
)

Device set to use cuda:0


## Creating the extended prompt
To create the prompt we use the result from query the Vector Database  and the sentence introduced by the user.

The prompt have two parts, the **relevant context** that is the information recovered from the database and the **user's question**.

We only need to join the two parts together to create the prompt that we are going to send to the model.

You can limit the lenght of the context passed to the model, because we can get some Memory problems with one of the datasets that contains a realy large text in the document part.

In [17]:
question = "Can I buy a Toshiba laptop?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])
#context = context[0:5120]
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"
prompt_template

"Relevant context: #['The MIT AI Hardware Program is a new academia and industry collaboration aimed at defining and developing translational technologies in hardware and software for the AI and quantum age. A collaboration between the MIT School of Engineering and MIT Schwarzman College of Computing, involving the Microsystems Technologies Laboratories and programs and units in the college, the cross-disciplinary effort aims to innovate technologies that will deliver enhanced energy efficiency systems for cloud and edge computing.', '“A sharp focus on AI hardware manufacturing, research, and design is critical to meet the demands of the world’s evolving devices, architectures, and systems,” says Anantha Chandrakasan, dean of the MIT School of Engineering and Vannevar Bush Professor of Electrical Engineering and Computer Science. “Knowledge-sharing between industry and academia is imperative to the future of high-performance computing.”', 'Based on use-inspired research involving mater

Now all that remains is to send the prompt to the model and wait for its response!


In [18]:
lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


Relevant context: #['The MIT AI Hardware Program is a new academia and industry collaboration aimed at defining and developing translational technologies in hardware and software for the AI and quantum age. A collaboration between the MIT School of Engineering and MIT Schwarzman College of Computing, involving the Microsystems Technologies Laboratories and programs and units in the college, the cross-disciplinary effort aims to innovate technologies that will deliver enhanced energy efficiency systems for cloud and edge computing.', '“A sharp focus on AI hardware manufacturing, research, and design is critical to meet the demands of the world’s evolving devices, architectures, and systems,” says Anantha Chandrakasan, dean of the MIT School of Engineering and Vannevar Bush Professor of Electrical Engineering and Computer Science. “Knowledge-sharing between industry and academia is imperative to the future of high-performance computing.”', 'Based on use-inspired research involving materi

# **Editting context + prompt (RAG- style, dataset aware)**

**Current problem:**
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])

this turns each retrieved document ito literal <doc text> string;

It **does not inlude titles**/topics clearly

The question "Can I buy a Toshiba laptop? **is not related to the dataset**

And prompt does not tell specifically to the model to:
**only use context**
> Add blockquote





In [31]:
# 1) build clean context with topics (titles)
retrieved_docs = results["documents"][0]
retrieved_meta = results["metadatas"][0]

context_blocks = []
for doc, meta in zip(retrieved_docs, retrieved_meta):
    title = meta.get("topic") or meta.get("Article Header") or "Unknown title"
    context_blocks.append(f"TITLE: {title}\nCONTENT: {doc}")

context = "\n\n---\n\n".join(context_blocks)

# 2) dataset-fit question
question = "What recent MIT AI News articles discuss energy-efficient AI hardware?"

# 3) prompt tuned for news RAG
MAX_CONTEXT_CHARS = 2000  # start small, increase if safe

context_short = context[:MAX_CONTEXT_CHARS]

prompt_template = f"""
You are a helpful assistant answering questions about MIT AI News.
Use ONLY the context below. If the answer isn't there, say you don't know.

Context:
{context_short}

Question: {question}

Answer in 3–6 sentences and list article titles used.
""".strip()


Model 2 = Different HF model (small + strong)


In [32]:
import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id_a = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_a = AutoTokenizer.from_pretrained(model_id_a)
model_a = AutoModelForCausalLM.from_pretrained(
    model_id_a,
    torch_dtype=torch.float16,
    device_map="auto"
)
model_a.config.use_cache = False

pipe_a = pipeline(
    "text-generation",
    model=model_a,
    tokenizer=tokenizer_a,
    max_new_tokens=80,
    temperature=0.2,
    do_sample=True
)
print("pipe_a ready (TinyLlama fp16)")



tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cuda:0


pipe_a ready (TinyLlama fp16)


In [33]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id_dolly = "databricks/dolly-v2-3b"

tokenizer_d = AutoTokenizer.from_pretrained(model_id_dolly)
model_d = AutoModelForCausalLM.from_pretrained(model_id_dolly).to("cpu")
model_d.config.use_cache = False  # still reduces RAM spikes

pipe_d = pipeline(
    "text-generation",
    model=model_d,
    tokenizer=tokenizer_d,
    max_new_tokens=80,
    temperature=0.2,
    do_sample=True,
    device=-1   # CPU
)

out_dolly = pipe_d(prompt_template)[0]["generated_text"]
print("Dolly (CPU):\n", out_dolly)


Device set to use cpu


Dolly (CPU):
 You are a helpful assistant answering questions about MIT AI News.
Use ONLY the context below. If the answer isn't there, say you don't know.

Context:
TITLE: New program bolsters innovation in next-generation artificial intelligence hardware 
CONTENT: ['The MIT AI Hardware Program is a new academia and industry collaboration aimed at defining and developing translational technologies in hardware and software for the AI and quantum age. A collaboration between the MIT School of Engineering and MIT Schwarzman College of Computing, involving the Microsystems Technologies Laboratories and programs and units in the college, the cross-disciplinary effort aims to innovate technologies that will deliver enhanced energy efficiency systems for cloud and edge computing.', '“A sharp focus on AI hardware manufacturing, research, and design is critical to meet the demands of the world’s evolving devices, architectures, and systems,” says Anantha Chandrakasan, dean of the MIT School of

In [34]:
print("TinyLlama output:\n", out_a)
print("\nDolly output:\n", out_dolly)


TinyLlama output:
 You are a helpful assistant answering questions about MIT AI News.
Use ONLY the context below. If the answer isn't there, say you don't know.

Context:
TITLE: New program bolsters innovation in next-generation artificial intelligence hardware 
CONTENT: ['The MIT AI Hardware Program is a new academia and industry collaboration aimed at defining and developing translational technologies in hardware and software for the AI and quantum age. A collaboration between the MIT School of Engineering and MIT Schwarzman College of Computing, involving the Microsystems Technologies Laboratories and programs and units in the college, the cross-disciplinary effort aims to innovate technologies that will deliver enhanced energy efficiency systems for cloud and edge computing.', '“A sharp focus on AI hardware manufacturing, research, and design is critical to meet the demands of the world’s evolving devices, architectures, and systems,” says Anantha Chandrakasan, dean of the MIT Scho

Models Tested

Model A: TinyLlama-1.1B-Chat (fp16, GPU)

Model B: Dolly-v2-3B (CPU — GPU 4-bit unavailable in this environment)


Groundedness (Use of Provided Context)

TinyLlama:
✔ Stays close to the context
✔ Repeats specific quotes correctly
✔ Does not hallucinate new article names

Dolly:
✔ Also stays inside the context
✖ But truncates early
✖ Repeats context text instead of answering the question directly

Winner: TinyLlama (more complete + focused)

Ability to Answer the Question

TinyLlama:
➤ Describes the MIT AI Hardware Program and its energy-efficiency focus
➤ Gives multiple sentences covering engineering challenges, collaboration, and goals
➤ Nearly completes the answer before cutting off

Dolly:
➤ Mostly repeats the first paragraph of the context
➤ Never actually answers the question (“Which articles?”)
➤ Cuts off even earlier

Winner: TinyLlama (closest to a full answer)

Conciseness & Structure

TinyLlama:
➤ Produces paragraphs
✖ Does not list article titles as bullets (instruction followed halfway)

Dolly:
✖ Produces repeated context instead of structured answer
✖ No title list, no conclusion

Winner: TinyLlama

Hallucinations

TinyLlama: No hallucinations detected

Dolly: No major hallucinations, but lacks relevance

Winner: Draw, both safe

Interpretation

TinyLlama is a smaller model but is much better instruction-tuned than Dolly-v2-3B.

Dolly is an older model (2023) and behaves like it was trained on loosely formatted instruction data → it often repeats context instead of reasoning with it.

TinyLlama uses newer conversational tuning → better adherence to prompt structure.



Conclusion (copy for assignment)

In this experiment I compared two different HuggingFace models for RAG-style question answering: TinyLlama-1.1B-Chat and Dolly-v2-3B. Even though Dolly has more parameters, TinyLlama produced clearer, more complete, and more instruction-aligned responses. Dolly tended to repeat context verbatim and often failed to answer the question directly. TinyLlama showed better grounding, better summarization, and stronger adherence to the structured prompt. This demonstrates that model size alone does not determine performance; quality of instruction tuning and model architecture matters significantly more.